# 정적·동적 병렬 실행 — 실습본

원본: `src/raw/Chapter02/03-langgraph-advanced/04_parallel_execution.py`  
슬라이드: `docs/raw/Chapter02/day01/3-4-langgraph-advanced.pdf`

## 학습 목표

- Fan-out/Fan-in과 Send·reducer
- 원본 코드를 CodeMate와 함께 읽고 실행 결과를 점검한다.


## 코드 흐름

정적: START → pros/cons → summary. 동적: START → Send(worker)들 → results

**읽을 때 확인할 점:** 병렬 작업의 완료 순서는 달라질 수 있다. Annotated reducer가 worker 결과를 합친다.


## 실행 전 확인

여러 LLM 호출이 발생하며 출력 순서는 달라질 수 있다.

아래 원본 코드 셀은 실행을 직접 선택해야 하는 `manual-run` 셀입니다. 실행 준비 셀은 프로젝트 루트를 찾고 원본 `.py`의 경로를 `__file__`에 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "Chapter02").is_dir()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError("저장소 안에서 이 노트북을 열어 주세요.")
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root / "src"))
__file__ = str(_repo_root / "src/raw/Chapter02/03-langgraph-advanced/04_parallel_execution.py")


## 원본 코드

다음 셀에는 원본 `.py` 전체가 그대로 들어 있습니다.


In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Send
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


# ── 패턴 1: 정적 병렬 (Fan-out → Fan-in) ────────────────────────────────────
# 고정된 수의 노드를 병렬로 실행할 때 사용한다.
# 각 분기가 State에 결과를 append하고, 이후 하나의 노드가 합쳐서 처리한다.

class StaticState(TypedDict):
    topic: str
    pros: str
    cons: str
    final: str


def pros_node(state: StaticState) -> dict:
    result = llm.invoke(f'{state["topic"]}의 장점 3가지를 간결하게 나열해줘.')
    return {'pros': result.content}


def cons_node(state: StaticState) -> dict:
    result = llm.invoke(f'{state["topic"]}의 단점 3가지를 간결하게 나열해줘.')
    return {'cons': result.content}


def summary_node(state: StaticState) -> dict:
    print(state['cons'], state['pros'])
    result = llm.invoke(
        f'장점:\n{state["pros"]}\n\n단점:\n{state["cons"]}\n\n위 내용을 바탕으로 한 문장 결론을 내려줘.'
    )
    return {'final': result.content}


static_builder = StateGraph(StaticState)
static_builder.add_node('pros', pros_node)
static_builder.add_node('cons', cons_node)
static_builder.add_node('summary', summary_node)

# TODO: START에서 pros/cons를 병렬로 실행하고 summary에서 합쳐 END로 이어 주세요.
static_builder.add_edge(START, 'pros')
static_builder.add_edge(START, 'cons')  # pros, cons 동시 시작
# 만약, 평소와 같이 summary 노드로 진입하는 간선이 1개만 있다면,
    # 당연하게도, 그 1개의 시작지점 노드 (pros)의 결과가 나오는 순간
    # 바로 summary노드가 작업을 시작 하겠지만
static_builder.add_edge('pros', 'summary') 
static_builder.add_edge('cons', 'summary')  # summary 노드로 진입하는 노드가 2개라면
                                            # 두 작업자의 작업이 모두 끝나야 summary가 실행
static_builder.add_edge('summary', END)
# TODO: 정적 병렬 그래프를 컴파일해 주세요.
static_app = static_builder.compile()

print('=== 정적 병렬 (Fan-out → Fan-in) ===')
result = static_app.invoke({'topic': 'Remote Work', 'pros': '', 'cons': '', 'final': ''})
print(f'장점: {result["pros"][:80]}...')
print(f'단점: {result["cons"][:80]}...')
print(f'결론: {result["final"]}')
print()


# ── 패턴 2: 동적 병렬 (Send API) ────────────────────────────────────────────
# 실행 시점에 작업 수가 결정될 때 사용한다.
# Send(노드명, state)를 리스트로 반환하면 LangGraph가 해당 노드를 병렬로 실행한다.

class DynamicState(TypedDict):
    queries: list[str]
    results: Annotated[list[str], lambda a, b: a + b]   # 결과를 누적


class WorkerState(TypedDict):
    query: str
    results: Annotated[list[str], lambda a, b: a + b]


def worker_node(state: WorkerState) -> dict:
    """각 쿼리를 독립적으로 처리하는 워커 노드."""
    result = llm.invoke(f'"{state["query"]}"에 대해 한 문장으로 답해줘.')
    return {'results': [f'[{state["query"]}] {result.content}']}


def fan_out(state: DynamicState) -> list[Send]:
    """쿼리 수만큼 worker_node를 병렬로 생성해 보낸다."""
    # TODO: queries 수만큼 Send('worker', ...) 목록을 만들어 주세요.
    # temp = []
    # for q in state['queries']:
        # temp.apppend(Send('worker', {'query': q, 'results': []}))
        # Send('worker', {'query': q, 'results': []})
    # python list comprehension
    # numbers = [num * 2 for num in [1, 2, 3, 4, 5]]
    return [
        Send('worker', {'query': q, 'results': []})
        for q in state['queries']
    ]


dynamic_builder = StateGraph(DynamicState)
dynamic_builder.add_node('worker', worker_node)
# 기존의 간선 형성 방식
# dynamic_builder.add_edge(START, 'worker')
# TODO: START에서 fan_out으로 worker를 동적 생성하고 worker 이후 END로 연결해 주세요.
dynamic_builder.add_conditional_edges(START, fan_out, ['worker'])
dynamic_builder.add_edge('worker', END)
# TODO: 동적 병렬 그래프를 컴파일해 주세요.
dynamic_app = dynamic_builder.compile()

print('=== 동적 병렬 (Send API) ===')
# 그래프를 설계하는 시점에서 애초에 질문을 리스트 형식으로 받을거야.
# 그래서, 동시에 여러개의 질문에 답변할 수 있도록 처리할거야.
queries = ['LangGraph란?', 'RAG란?', 'MCP란?']
result2 = dynamic_app.invoke({'queries': queries, 'results': []})
for r in result2['results']:
    print(f'  {r}')


## CodeMate와 함께 검토

> 두 정적 노드가 같은 필드를 동시에 쓰면 생길 문제와 Send 결과가 reducer로 합쳐지는 과정을 설명해 줘. 원본은 바꾸지 말고 검토 내용을 먼저 말해 줘.

제안된 변경은 어떤 문제를 해결하는지, 어떤 결과로 확인할지 설명해 보세요.

## 확인 질문

1. summary는 언제 실행되는가?
2. results에 reducer가 필요한 이유는?
